# Preparation

In [ ]:
# You can download data like this (below you can test it)
# !wget -O wikitext-103.zip "https://www.dropbox.com/scl/fi/e6oqpx6iuos7kn9m139z7/wikitext-103-raw-v1.zip?rlkey=81evwbaqfkxtckj8zhks7yied&st=6ept2pdm&dl=0"
# !unzip wikitext-103.zip

In [ ]:
# main libs i used
# import sys
# !{sys.executable} -m pip install torch transformers einops sortedcontainers pandas plotly nbformat

In [ ]:
# Use this for flash attention (works on 3090)
# !{sys.executable} -m pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

In [4]:
import torch
from tqdm import tqdm
import time

In [ ]:
from config import SKIP_LENGTH, train_paths, tokenizer
from data import BaseDataset

In [6]:
test_ds = BaseDataset(train_paths, tokenizer, line_limits=100)
print(f"Sucessfully loaded {len(test_ds.lines)} lines, {sum(len(l) for l in test_ds.lines) * 4 / (1024 * 1024 * 1024)} GB total")
del test_ds

Sucessfully loaded 100 lines, 0.00015681609511375427 GB total


# What `batch_size`?

In [7]:
import subprocess, sys

def find_max_bs(scenario, flash, lo=1, hi=1024):
    result = 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r = subprocess.run(
            [sys.executable, "check_oom.py", str(scenario), str(mid), str(flash)],
            capture_output=True, text=True, timeout=120
        )
        if "OK" in r.stdout:
            result = mid
            lo = mid + 1
        else:
            print(f"s={scenario} flash={flash} bs={mid}: {r.stdout.strip()} | {r.stderr.strip()[-200:]}")
            hi = mid - 1
    return result

In [8]:
for scenario, scenario_name in enumerate(["batched", "packed_flattened"]):
    for flash in [0, 1]:
        bs = find_max_bs(scenario, flash)
        print(f"{scenario_name}, flash={bool(flash)}: max_bs={bs}")

s=0 flash=0 bs=512: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
s=0 flash=0 bs=448: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
s=0 flash=0 bs=440: OOM | 
batched, flash=False: max_bs=439
s=0 flash=1 bs=768: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
s=0 flash=1 bs=640: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
s=0 flash=1 bs=576: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
s=0 flash=1 bs=544: OOM | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enab

In [9]:
# I got next results on RTX 3090:
# batched_bs = 439
# packed_flattened_bs = 22
# batched_bs_flash = 515
# packed_flattened_bs_flash = 503

# Let's decrease them a bit and round:
batched_bs = 400
packed_flattened_bs = 20
batched_bs_flash = 480
packed_flattened_bs_flash = 464

# Runs

In [17]:
import subprocess, sys, json

runs = [
    # default
    ("static_pad", batched_bs, 1, False),
    ("dynamic_pad", batched_bs, 1, False),
    # default + flash
    ("static_pad", batched_bs_flash, 1, True),
    ("dynamic_pad", batched_bs_flash, 1, True),

    # packed
    ("packed_flattened_pad", packed_flattened_bs, 1, False),

    # packed + flash
    ("packed_flattened_pad_flash", packed_flattened_bs_flash, 1, True),
]

for nb in [10, 100, 300, SKIP_LENGTH]:
    runs.append(("binned_pad", batched_bs, nb, False))
    runs.append(("binned_pad", batched_bs_flash, nb, True))

all_results = []
for strategy, bs, nb, flash in runs:
    r = subprocess.run(
        [sys.executable, "run_bench.py", strategy, str(bs), str(nb), str(int(flash))],
        capture_output=True, text=True, timeout=600
    )
    try:
        data = json.loads(r.stdout)
        all_results.append((data["times"], data["peak_mem"]))
        print(f"OK: {strategy} bs={bs} flash={flash}, nb={nb}")
    except:
        print(f"FAIL: {strategy} bs={bs} flash={flash}, nb={nb}")
        print(f"  stdout: {r.stdout[-300:]}")
        print(f"  stderr: {r.stderr[-300:]}")


OK: static_pad bs=400 flash=False, nb=1
OK: dynamic_pad bs=400 flash=False, nb=1
OK: static_pad bs=480 flash=True, nb=1
OK: dynamic_pad bs=480 flash=True, nb=1
OK: packed_flattened_pad bs=20 flash=False, nb=1
OK: packed_flattened_pad_flash bs=464 flash=True, nb=1
OK: binned_pad bs=400 flash=False, nb=10
OK: binned_pad bs=480 flash=True, nb=10
OK: binned_pad bs=400 flash=False, nb=100
OK: binned_pad bs=480 flash=True, nb=100
OK: binned_pad bs=400 flash=False, nb=300
OK: binned_pad bs=480 flash=True, nb=300
OK: binned_pad bs=400 flash=False, nb=640
OK: binned_pad bs=480 flash=True, nb=640


# Analysis

In [18]:
import pandas as pd
all_times = []
all_peaks = {}

for times, peak_mem in all_results:
    all_times.extend(times)
    key = (times[0]["strategy"], times[0]["bs"], times[0]["nb"])
    all_peaks[key] = peak_mem

df = pd.DataFrame(all_times)
df["tokens_in_batch"] = df.groupby(["strategy", "bs", "nb"])["total_tokens"].diff().fillna(df["total_tokens"])
df["tokens_per_sec"] = df["tokens_in_batch"] / df["time"]

summary = df.groupby(["strategy", "bs", "nb", "flash"]).agg(
    median_time=("time", "median"),
    q25_time=("time", lambda x: x.quantile(0.25)),
    q75_time=("time", lambda x: x.quantile(0.75)),
    median_tps=("tokens_per_sec", "median"),
).reset_index()

summary["peak_mem_gb"] = summary.apply(lambda r: all_peaks.get((r["strategy"], r["bs"], r["nb"]), None), axis=1)
summary.sort_values("median_tps")

,strategy,bs,nb,flash,median_time,q25_time,q75_time,median_tps,peak_mem_gb
10,packed_flattened_pad,20,1,False,0.887204,0.886505,0.888372,14427.348024,13.960652
12,static_pad,400,1,False,2.425601,2.423867,2.426857,16405.316626,18.035256
13,static_pad,480,1,True,2.011518,2.010550,2.012430,23719.174154,21.429383
8,dynamic_pad,400,1,False,1.610782,1.408312,1.834936,24151.064478,18.034119
9,dynamic_pad,480,1,True,1.427450,1.264047,1.592738,32212.452298,21.428761
0,binned_pad,400,10,False,0.214372,0.065670,0.627955,81363.269492,13.180027
1,binned_pad,400,100,False,0.076729,0.029948,0.247107,88157.857288,7.100508
3,binned_pad,400,640,False,0.021882,0.010450,0.039023,88596.764903,1.974119
2,binned_pad,400,300,False,0.048447,0.020934,0.110077,89574.546577,3.840313
4,binned_pad,480,10,True,0.212502,0.073950,0.567977,101028.481479,13.300334


In [19]:
import json

with open("benchmark_full.json", "w") as f:
    json.dump([{"times": t, "peak_mem": p} for t, p in all_results], f)

df.to_csv("benchmark_raw.csv", index=False)
summary.to_csv("benchmark_summary.csv", index=False)

# Visualisation

In [1]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go

pio.renderers.default = "notebook"

In [2]:
summary = pd.read_csv("benchmark_summary.csv")
best = summary.loc[summary.groupby(["strategy", "flash"])["median_tps"].idxmax()]
sorted_df = best.sort_values("median_tps", ascending=True)

In [13]:
short_labels = {
    "static_pad": "Static",
    "dynamic_pad": "Dynamic",
    "binned_pad": "Binned",
    "packed_flattened_pad": "Packed Flat",
    "packed_flattened_pad_flash": "Packed Flat",
}

colors = {
    "static_pad": "#E74C3C",
    "dynamic_pad": "#E67E22",
    "binned_pad": "#2ECC71",
    "packed_flattened_pad": "#3498DB",
    "packed_flattened_pad_flash": "#9B59B6",
}

flash_suffix = sorted_df["flash"].map({True: " ⚡", False: ""})
labels = sorted_df["strategy"].map(short_labels) + flash_suffix + " (bs=" + sorted_df["bs"].astype(str) + ")"

fig = go.Figure()
fig.add_trace(go.Bar(
    y=labels, x=sorted_df["median_tps"],
    orientation="h",
    marker_color=[colors.get(s, "#666") for s in sorted_df["strategy"]],
    text=sorted_df["median_tps"].apply(lambda x: f"{x:,.0f}"),
    textposition="outside",
    textfont=dict(size=18),
))
fig.update_layout(
    height=500, width=1000,
    template="plotly_dark",
    title=dict(text="Throughput (tokens/sec) — GPT-2 · RTX 3090 · bf16", font=dict(size=22)),
    margin=dict(l=250, r=180, b=60),
    xaxis_title="tokens/sec        ⚡ = Flash Attention backend",
    xaxis=dict(title_font=dict(size=16), tickfont=dict(size=14)),
    yaxis=dict(tickfont=dict(size=16)),
)
fig.update_traces(cliponaxis=False)
fig.show()

In [12]:
# it needs kaleido which depends on chrome, so I'll make it locally
with open("benchmark_throughput.svg", "wb") as f:
    f.write(fig.to_image(format="svg", width=2400, height=1000))